## 1. Установка зависимостей и проверка GPU


In [ ]:
%pip -q install "torch==2.9.1" "u8darts[torch]>=0.38,<0.41" "vmdpy==0.2" "scikit-learn>=1.3" pandas numpy matplotlib joblib

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Для ускорения выберите Runtime → Change runtime type → T4 GPU')


## 2. Получение проекта из fork

Перед запуском сохраните и опубликуйте изменения репозитория в выбранной ветке. Исходники используются напрямую, без runtime-патчей.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Yuri-ga1/VMD-TiDE.git"
REPO_BRANCH = "master"
project = Path('/content/VMD-TiDE')
if not project.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(project)], check=True)
else:
    print('Используется существующая копия; для обновления выполните git pull после проверки локальных изменений.')
print('Commit:', subprocess.check_output(['git', '-C', str(project), 'rev-parse', 'HEAD'], text=True).strip())
if not (project / 'experiments/three_models.py').is_file():
    raise FileNotFoundError('В выбранной ветке нет experiments/three_models.py. Сначала опубликуйте изменения fork.')


## 3. Подключение Google Drive и датасета

Используется `Exam/dataset.csv`; модели сохраняются непосредственно в `Exam/models`. PostgreSQL и ZIP не требуются.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
EXAM_DIR = Path('/content/drive/MyDrive/Учеба/Магистратура/BigData/Exam')
MODELS_DIR = EXAM_DIR / 'models'
CSV_PATH = EXAM_DIR / 'dataset.csv'
if not CSV_PATH.is_file():
    raise FileNotFoundError(f'Датасет не найден: {CSV_PATH}')
print('Проект:', project)
print('CSV:', CSV_PATH)
print('Каталог моделей:', MODELS_DIR)


## 4. Настройка и проверка Train / Validation / Test

Сохраняются 12 моделей A, 12 B и одна SN. Максимум 50 эпох;
Early Stopping отслеживает `val_loss` (patience=10, min_delta=0.0001).
Валидация TiDE и Fusion Ridge используют 1986-09-01 — 2008-12-31.
VMD для валидации заканчивается 2008-12-31 и не видит Test.
Лучший checkpoint загружается перед сохранением каждого TiDE.

`FORCE_TRAIN=False` возобновляет совместимый запуск. Старые веса без Early Stopping
сохраняются: при несовместимости явно задайте новый `EXPERIMENT_NAME`, например
`VMD_TiDE_ABC_h7_train1954_test2009_early_stopping`. Автоматического переобучения нет.


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# ============================================================
# 1. Подключение проекта
# ============================================================

os.chdir(project)

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

import experiments.three_models as tm

from experiments.three_models import Config, load_data, MODES


# ============================================================
# 2. Параметры обучения
# ============================================================

HORIZON = 7
EPOCHS = 50          # Максимальное количество эпох
MAX_ORIGINS = None   # None = все доступные точки прогноза
STRIDE = 27          # Шаг между прогнозами в днях

FORCE_TRAIN = False  # Использовать сохранённые веса, если есть


# ============================================================
# 3. Временные интервалы
# ============================================================

TRAIN_START = "1954-01-02"
TRAIN_END = "1986-08-31"

VAL_START = "1986-09-01"
VAL_END = "2008-12-31"

TEST_START = "2009-01-01"


# ============================================================
# 4. Определение последней доступной даты
# ============================================================

# CSV_PATH задан в ячейке подключения Drive: Exam/dataset.csv

raw = pd.read_csv(CSV_PATH)

raw["date"] = pd.to_datetime(
    raw[["Year", "Month", "Day"]].rename(
        columns={
            "Year": "year",
            "Month": "month",
            "Day": "day"
        }
    )
)

# Учитываем только даты с доступными F10.7 и SN
valid = (
    raw["F10.7adj"].notna()
    & (raw["F10.7adj"] > 0)
    & raw["SN"].notna()
    & (raw["SN"] >= 0)
)

LAST_DATE = raw.loc[valid, "date"].max()

if pd.isna(LAST_DATE):
    raise ValueError(
        "В датасете не найдены совместные наблюдения F10.7 и SN"
    )

TEST_END = LAST_DATE.strftime("%Y-%m-%d")


# ============================================================
# 5. Папка сохранения моделей на Google Drive
# ============================================================

# Отдельный эксперимент для нового временного диапазона
EXPERIMENT_NAME = (
    f"VMD_TiDE_ABC_h{HORIZON}_train1954_test2009"
)

DRIVE_OUTPUT = MODELS_DIR / EXPERIMENT_NAME

# Старые веса не переименовываются и не перезаписываются автоматически.


# ============================================================
# 6. Конфигурация модели
# ============================================================

cfg = Config(
    csv_path=str(CSV_PATH),
    output_dir=str(DRIVE_OUTPUT),

    target="F10.7adj",

    start=TRAIN_START,
    train_end=TRAIN_END,

    # В текущем коде fusion_end обозначает конец
    # валидационного интервала для Ridge-модели C.
    fusion_end=VAL_END,

    test_end=TEST_END,

    horizon=HORIZON,
    context=126,

    epochs=EPOCHS,
    stride=STRIDE,
    max_origins=MAX_ORIGINS,

    vmd_history=2048,
    batch_size=256,
    random_seed=42,
)


# ============================================================
# 7. Загрузка и подготовка данных
# ============================================================

df = load_data(cfg)
# Проверка до обучения и записи config.json. Старые веса без Early Stopping
# требуют явно выбранного нового EXPERIMENT_NAME; FORCE_TRAIN их не перезапишет.
tm._check_config(cfg, initialize=False) if (DRIVE_OUTPUT / 'training_fingerprint.txt').exists() else None
if DRIVE_OUTPUT.exists() and any(DRIVE_OUTPUT.iterdir()) and not (DRIVE_OUTPUT / 'training_fingerprint.txt').exists():
    raise RuntimeError('В каталоге прежние результаты. Выберите новый EXPERIMENT_NAME для Early Stopping.')
print('Training protocol:', tm.TRAINING_PROTOCOL)


# ============================================================
# 8. Проверка разбиения
# ============================================================

train = df.loc[TRAIN_START:TRAIN_END]
val = df.loc[VAL_START:VAL_END]
test = df.loc[TEST_START:TEST_END]

assert len(train) > 0, "Train пустой"
assert len(val) > 0, "Validation пустой"
assert len(test) > 0, "Test пустой"

assert train.index.max() < val.index.min()
assert val.index.max() < test.index.min()

summary = pd.DataFrame({
    "Dataset": ["Train", "Validation", "Test"],
    "Start": [
        train.index.min().date(),
        val.index.min().date(),
        test.index.min().date()
    ],
    "End": [
        train.index.max().date(),
        val.index.max().date(),
        test.index.max().date()
    ],
    "Days": [
        len(train),
        len(val),
        len(test)
    ],
    "Missing F10.7": [
        train["f107_original"].isna().sum(),
        val["f107_original"].isna().sum(),
        test["f107_original"].isna().sum()
    ],
    "Missing SN": [
        train["sn_original"].isna().sum(),
        val["sn_original"].isna().sum(),
        test["sn_original"].isna().sum()
    ]
})

print("=" * 65)
print("DATASET SPLIT")
print("=" * 65)

display(summary)

print("\nModel components:", len(MODES))
print("Forecast horizon:", HORIZON)
print("Maximum epochs:", EPOCHS)
print("Forecast stride:", STRIDE)

print("\nLast available observation:", TEST_END)
print("Save destination:", cfg.output_dir)


# ============================================================
# 9. Проверка ранее сохранённых моделей
# ============================================================

print("\nSaved model components:")

for variant in ("A", "B", "SN"):
    directory = DRIVE_OUTPUT / "weights" / variant

    count = (
        len(list(directory.glob("*.pt")))
        if directory.exists()
        else 0
    )

    expected = 1 if variant == "SN" else len(MODES)

    print(f"{variant}: {count}/{expected}")


print("\nConfiguration completed successfully.")

## 5. Последовательное обучение A → B → SN → C

Каждый компонент TiDE сохраняется сразу после `fit()`. После завершения 12 компонентов A появляется `A_complete.json`; после B — `B_complete.json`; после SN — `SN_complete.json`. C (Ridge) записывается после обучения на существующем Validation / Fusion периоде. Если Colab прервётся, вновь выполните ячейки сверху вниз, оставив `FORCE_TRAIN=False` и прежнюю конфигурацию.

In [ ]:
from experiments.three_models import run
metrics, test_predictions = run(cfg, force_train=FORCE_TRAIN)
display(metrics.round(4))

## 6. Метрики по дням горизонта прогноза

In [ ]:
from experiments.three_models import evaluate
from IPython.display import display
lead_rows = []
for lead, group in test_predictions.groupby('lead'):
    scores = evaluate(group)
    for model, row in scores.iterrows():
        lead_rows.append({'day_ahead': lead, 'model': model, **row.to_dict()})
by_lead = pd.DataFrame(lead_rows)
display(by_lead.pivot(index='day_ahead', columns='model', values='MAE').round(3))
if not (DRIVE_OUTPUT / 'metrics_by_lead.csv').exists():
    by_lead.to_csv(DRIVE_OUTPUT / 'metrics_by_lead.csv', index=False)
print('Metrics saved:', DRIVE_OUTPUT / 'metrics_by_lead.csv')

## 7. Проверка сохранённых моделей на Google Drive

Веса хранятся в `Exam/models/VMD_TiDE_ABC_h7_train1954_test2009/weights/{A,B,SN}/`, Ridge — `fusion_ridge.joblib`, PNG/CSV — в том же каталоге эксперимента. Никакая дополнительная упаковка не нужна.

In [ ]:
for variant in ('A', 'B', 'SN', 'C'):
    marker = DRIVE_OUTPUT / f'{variant}_complete.json'
    print(f'{variant}:', 'Сохранена' if marker.is_file() else 'Пока не завершена')
print('Всего весов TiDE:', len(list((DRIVE_OUTPUT / 'weights').rglob('*.pt'))))
print('Каталог результатов:', DRIVE_OUTPUT)
import json
training_rows = []
for path in sorted((DRIVE_OUTPUT / 'weights').rglob('*.training.json')):
    info = json.loads(path.read_text())
    training_rows.append({'component': f'{path.parent.name}/{path.stem}',
                          'best_epoch': info['best_epoch'],
                          'epochs_completed': info['epochs_completed'],
                          'best_val_loss': info['best_val_loss']})
display(pd.DataFrame(training_rows))


## 8. Загрузка без переобучения и ретроспективный прогноз из Test

Эту ячейку можно выполнять после настройки, пропустив обучение и расчёт общих метрик. График и таблица выводятся только в Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

from experiments.three_models import (
    load_models,
    load_data,
    forecast_at
)

# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

# Последний день, данные которого известны моделям
FORECAST_ORIGIN = "2020-06-01"

# Сколько дней наблюдений показывать до прогноза
HISTORY_DAYS = 60

# Длительность прогноза определяется обученной моделью
FORECAST_DAYS = cfg.horizon


# ============================================================
# 2. ЗАГРУЗКА ОБУЧЕННЫХ МОДЕЛЕЙ
# ============================================================

print("Загрузка обученных моделей...")

models = load_models(cfg)

ridge = joblib.load(
    Path(cfg.output_dir) / "fusion_ridge.joblib"
)

print("Модели A, B, C загружены.")


# ============================================================
# 3. ЗАГРУЗКА ДАННЫХ
# ============================================================

df = load_data(cfg)

origin = pd.Timestamp(FORECAST_ORIGIN)

future_dates = pd.date_range(
    start=origin + pd.Timedelta(days=1),
    periods=FORECAST_DAYS,
    freq="D"
)

# Проверяем, что прогноз относится к тестовой выборке
test_start = pd.Timestamp(cfg.fusion_end) + pd.Timedelta(days=1)
test_end = pd.Timestamp(cfg.test_end)

if origin < test_start:
    raise ValueError(
        f"Выбранная дата раньше тестового периода: {test_start.date()}"
    )

if future_dates[-1] > test_end:
    raise ValueError(
        f"Прогноз выходит за пределы тестовой выборки: "
        f"{test_end.date()}"
    )

print("Дата начала прогноза:", origin.date())
print("Прогнозируемый период:",
      future_dates[0].date(), "—", future_dates[-1].date())


# ============================================================
# 4. ПРОГНОЗ МОДЕЛЕЙ A И B
# ============================================================

# forecast_at использует историю не позднее origin
predictions = forecast_at(
    cfg=cfg,
    df=df,
    models=models,
    origin=origin
)

forecast_A = np.asarray(predictions["A"]).ravel()
forecast_B = np.asarray(predictions["B"]).ravel()

# Прогноз солнечных пятен
forecast_SN = np.asarray(predictions["SN"]).ravel()


# ============================================================
# 5. ПРОГНОЗ МОДЕЛИ C
# ============================================================

X_fusion = np.column_stack([
    forecast_A,
    forecast_SN
])

forecast_C = ridge.predict(X_fusion).ravel()


# ============================================================
# 6. ФАКТИЧЕСКИЕ НАБЛЮДЕНИЯ
# ============================================================

actual = df["f107_original"].reindex(
    future_dates
).to_numpy(dtype=float)

# Проверка наличия наблюдений
valid = np.isfinite(actual)

if not valid.any():
    raise ValueError(
        "На выбранном периоде отсутствуют реальные наблюдения F10.7"
    )


# ============================================================
# 7. ТАБЛИЦА СРАВНЕНИЯ
# ============================================================

comparison = pd.DataFrame({
    "Date": future_dates,
    "Observed": actual,
    "Model_A": forecast_A,
    "Model_B": forecast_B,
    "Model_C": forecast_C,
})

print("\nСравнение прогнозов с наблюдениями:")
display(comparison.round(2))


# ============================================================
# 8. РАСЧЁТ МЕТРИК
# ============================================================

metrics_rows = []

for name in ["Model_A", "Model_B", "Model_C"]:

    y_true = actual[valid]
    y_pred = comparison[name].to_numpy()[valid]

    metrics_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "R2": (
            r2_score(y_true, y_pred)
            if len(y_true) >= 2
            else np.nan
        )
    })

metrics_df = pd.DataFrame(metrics_rows).set_index("Model")

baseline_mae = metrics_df.loc["Model_A", "MAE"]

metrics_df["MAE improvement, %"] = (
    (baseline_mae - metrics_df["MAE"])
    / baseline_mae * 100
    if baseline_mae != 0
    else np.nan
)

print("\nМетрики выбранного периода:")
display(metrics_df.round(4))


# ============================================================
# 9. ИСТОРИЯ НАБЛЮДЕНИЙ
# ============================================================

history_start = origin - pd.Timedelta(
    days=HISTORY_DAYS - 1
)

history = df.loc[
    history_start:origin,
    "f107_original"
].dropna()


# ============================================================
# 10. ОБЩИЙ ГРАФИК
# ============================================================

fig, ax = plt.subplots(figsize=(15, 6))

# Исторические наблюдения
ax.plot(
    history.index,
    history.values,
    color="black",
    linewidth=1.8,
    label="F10.7 — наблюдения"
)

# Фактические значения в прогнозируемом периоде
ax.plot(
    future_dates,
    actual,
    color="black",
    linewidth=2.5,
    marker="o",
    label="F10.7 — фактические значения"
)

# Прогнозы моделей
colors = {
    "Model_A": "royalblue",
    "Model_B": "darkorange",
    "Model_C": "forestgreen"
}

labels = {
    "Model_A": "A — VMD-TiDE",
    "Model_B": "B — VMD-TiDE + SN",
    "Model_C": "C — Ensemble"
}

for name in ["Model_A", "Model_B", "Model_C"]:

    ax.plot(
        future_dates,
        comparison[name],
        marker="o",
        linewidth=2,
        color=colors[name],
        label=labels[name]
    )

# Начало прогноза
ax.axvline(
    origin,
    color="gray",
    linestyle="--",
    linewidth=1.5,
    label="Начало прогноза"
)

# Подсветка прогнозируемого периода
ax.axvspan(
    future_dates[0],
    future_dates[-1],
    color="gray",
    alpha=0.08
)

ax.set_title(
    f"Прогноз F10.7 на {FORECAST_DAYS} дней "
    f"от {origin.date()}"
)

ax.set_xlabel("Дата")
ax.set_ylabel("F10.7, sfu")

ax.legend()
ax.grid(alpha=0.3)

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()